# 07 — Yadav et al. (2026): Z-map vs ZZ-map QSVM, on 4 diabetes datasets

**Method (PeerJ CS):** QSVC with the unentangled Z feature map and the entangling ZZ feature map
(reps = 2), 80/20 split. Their benchmark was WDBC (breast cancer, out of scope here); only the
method is transferred. The hybrid QNN head is not repeated (notebook 05 covers VQCs).

**Protocols**
* `paper` — 80/20 stratified hold-out (seed 42), ungrouped, scaling on all rows, bandwidth 1.
* `clean` — grouped/stratified 5-fold × seeds {0, 1, 42}; models at bandwidth 1 **and** with the
  bandwidth chosen by inner 3-fold CV (balanced accuracy) on the training fold from
  {0.1, 0.25, 0.5, 1.0}. The outer test fold is never used for the choice.

In [1]:
import os, sys
from pathlib import Path
for p in [Path.cwd(), Path.cwd() / "notebooks2", Path("/content/notebooks2"), *Path("/kaggle/input").glob("*"), *Path("/kaggle/input").glob("*/notebooks2")]:
    if (p / "nb2_common.py").exists():
        sys.path.insert(0, str(p)); break
from nb2_common import *
from sklearn.svm import SVC
sanity_check_kernels()
DATA = load_all()

nb2_common loaded: numpy 2.4.6, sklearn 1.6.1
kernel self-test passed
pima     N=  768 d=  8 prevalence=0.349 unique=  768 grouped_cv=False  [diabetes]
sylhet   N=  520 d= 16 prevalence=0.615 unique=  251 grouped_cv=True  [diabetes (early stage)]


cdc      N= 2000 d= 21 prevalence=0.140 unique= 1995 grouped_cv=True  [diabetes or prediabetes (self-reported)]


hosp130  N= 2000 d= 44 prevalence=0.088 unique= 2000 grouped_cv=False  [readmitted within 30 days]


In [2]:
SCALES = [0.1, 0.25, 0.5, 1.0]
MAPS = {
    "Z-QSVM": lambda X, s: pauli_states(X, reps=2, scale=s, pairs=False),
    "ZZ-QSVM (linear)": lambda X, s: pauli_states(X, reps=2, scale=s, entangle="linear"),
    "ZZ-QSVM (full)": lambda X, s: pauli_states(X, reps=2, scale=s, entangle="full"),
}


def pick_scale(fmap, Xq, y, groups, seed):
    best, best_s = -1, 1.0
    for s in SCALES:
        K = state_kernel(fmap(Xq, s), fmap(Xq, s))
        scores = []
        for a, b in inner_splits(y, groups, seed, n_splits=3):
            p, _ = svm_precomputed(K[np.ix_(a, a)], y[a], K[np.ix_(b, a)], seed=seed)
            scores.append(balanced_accuracy_score(y[b], p))
        if np.mean(scores) > best:
            best, best_s = np.mean(scores), s
    return best_s


def make_method(tune):
    def yadav_method(Xtr, ytr, Xte, seed, groups_tr, X_prep):
        Xq_tr, Xq_te = prep(to_qubits, X_prep, Xtr, Xte, seed=seed)
        out = {}
        for name, fmap in MAPS.items():
            scales = [1.0] + ([pick_scale(fmap, Xq_tr, ytr, groups_tr, seed)] if tune else [])
            for j, s in enumerate(scales):
                Ptr, Pte = fmap(Xq_tr, s), fmap(Xq_te, s)
                label = name + (" bw=1" if j == 0 else " bw=inner-CV")
                out[label] = svm_precomputed(state_kernel(Ptr, Ptr), ytr, state_kernel(Pte, Ptr), seed=seed)
                if j == 1:
                    CHOSEN.append(dict(map=name, scale=s))
        Xs_tr, Xs_te = prep(impute_scale, X_prep, Xtr, Xte)
        rbf = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=seed).fit(Xs_tr, ytr)
        out["RBF-SVM"] = (rbf.predict(Xs_te), rbf.decision_function(Xs_te))
        return out
    return yadav_method


CHOSEN = []
rec = Recorder("P07_Yadav", "07_yadav", config=dict(
    method="Z / ZZ-linear / ZZ-full feature maps reps=2, QSVC C=1; bandwidth grid " + str(SCALES),
    paper_protocol="80/20 stratified hold-out seed 42, global scaling, bandwidth 1"))
print("paper protocol"); run_paper_holdout(rec, DATA, make_method(tune=False), test_size=0.2, global_prep=True)
print("clean protocol"); run_clean(rec, DATA, make_method(tune=True))
df = rec.save()

paper protocol


  paper  pima     test rows sharing a feature vector with train: 0


  paper  sylhet   test rows sharing a feature vector with train: 72


  paper  cdc      test rows sharing a feature vector with train: 1


  paper  hosp130  test rows sharing a feature vector with train: 0
clean protocol


  clean  pima     done in   35.2s


  clean  sylhet   done in   33.2s


  clean  cdc      done in  172.7s


  clean  hosp130  done in  123.7s
saved /content/notebooks2/results/07_yadav.csv  (436 rows)


In [3]:
pd.set_option("display.width", 220, "display.max_rows", 200)
summarise(df)

,dataset,protocol,model,accuracy,balanced_accuracy,recall,specificity,roc_auc,n_runs
0,cdc,clean,RBF-SVM,0.862 ± 0.017,0.531 ± 0.018,0.071 ± 0.039,0.99 ± 0.005,0.75 ± 0.037,15
1,cdc,clean,Z-QSVM bw=1,0.86 ± 0.016,0.513 ± 0.012,0.032 ± 0.029,0.995 ± 0.005,0.708 ± 0.05,15
2,cdc,clean,Z-QSVM bw=inner-CV,0.86 ± 0.016,0.513 ± 0.012,0.032 ± 0.029,0.994 ± 0.005,0.7 ± 0.046,15
3,cdc,clean,ZZ-QSVM (full) bw=1,0.861 ± 0.018,0.501 ± 0.002,0.001 ± 0.004,1.0 ± 0.0,0.635 ± 0.022,15
4,cdc,clean,ZZ-QSVM (full) bw=inner-CV,0.861 ± 0.018,0.527 ± 0.016,0.062 ± 0.034,0.991 ± 0.005,0.7 ± 0.041,15
5,cdc,clean,ZZ-QSVM (linear) bw=1,0.86 ± 0.018,0.506 ± 0.006,0.014 ± 0.012,0.998 ± 0.002,0.684 ± 0.029,15
6,cdc,clean,ZZ-QSVM (linear) bw=inner-CV,0.86 ± 0.015,0.52 ± 0.018,0.049 ± 0.037,0.992 ± 0.005,0.704 ± 0.046,15
7,cdc,paper,RBF-SVM,0.868 ± nan,0.527 ± nan,0.054 ± nan,1.0 ± nan,0.782 ± nan,1
8,cdc,paper,Z-QSVM bw=1,0.865 ± nan,0.518 ± nan,0.036 ± nan,1.0 ± nan,0.72 ± nan,1
9,cdc,paper,ZZ-QSVM (full) bw=1,0.86 ± nan,0.5 ± nan,0.0 ± nan,1.0 ± nan,0.613 ± nan,1


## Bandwidths picked by inner CV (counts over datasets × seeds × folds)

In [4]:
pd.DataFrame(CHOSEN).value_counts().rename("times chosen").reset_index()

,map,scale,times chosen
0,ZZ-QSVM (full),0.10,45
1,ZZ-QSVM (linear),0.10,39
2,Z-QSVM,0.25,18
3,Z-QSVM,0.10,18
4,ZZ-QSVM (linear),0.50,15
5,ZZ-QSVM (full),0.25,14
6,Z-QSVM,1.00,13
7,Z-QSVM,0.50,11
8,ZZ-QSVM (linear),0.25,6
9,ZZ-QSVM (full),0.50,1
